## **Exploratory Data Analysis**

#### **Dataset**

We decided to go on the adversarial QA dataset as it a very interesting one to evaluate the flan-T5 model, how it performs initially on an adversial dataset without fine-tuning and on different quantisation level, to see how quantisation affect performances. Then later on we will fine tune the model on the dataset in order to see how fine tuning affect the performance of the model on the different quantisation levels. 

In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer
device = "cuda:0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

In [23]:
model_id = "google/flan-t5-base"
ds = load_dataset("UCLNLP/adversarial_qa", "adversarialQA")

Let us explore the data set

In [24]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata'],
        num_rows: 30000
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata'],
        num_rows: 3000
    })
})

In [26]:
for dset in ds:
    n = len(ds[dset])
    n_empty = sum(1 for a in ds[dset]["answers"] if len(a["text"]) == 0)
    print(f"{dset:12s} {n:6d} rows | {n_empty:5d} rows without answer")

train         30000 rows |     0 rows without answer
validation     3000 rows |     0 rows without answer
test           3000 rows |  3000 rows without answer


As we can see the dataset is already split into train, validation and test sets. As mentioned in the documentation, https://huggingface.co/datasets/UCLNLP/adversarial_qa, there are no answer provided with the test set. We'll therefore ignore it, use the current validation set as our test set, and modify the train set to include 27k examples instead of 30k, keeping the 3k for a new validation set.

In [27]:
split = ds["train"].train_test_split(test_size=3000, seed=42)
train_ds = split["train"]
val_ds = split["test"]
test_ds = ds["validation"]
lengths = {"train":len(train_ds), "val":len(val_ds), "test":len(test_ds)}

ds = {"train":train_ds, "val":val_ds, "test":test_ds}

for name, dset in ds.items():
    n = len(dset)
    n_empty = sum(1 for answer in dset["answers"] if len(answer["text"]) == 0)
    print(f"{name:12s} {n:6d} rows | {n_empty:5d} rows without answer")

train         27000 rows |     0 rows without answer
val            3000 rows |     0 rows without answer
test           3000 rows |     0 rows without answer


Let us explore also the tokens length of the context + question that we will later use for the prompt of the model. We'll use for that the tokenizer of the model we chose for the project, which is flan-t5.

In [28]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

### Token lengths

Flan-T5 takes one single string as input, so we build the prompt by merging
`question` and `context` together. What we want to know here is how long that
merged string actually is once tokenized, and how long the answers are.

Two reasons for this:
- the prompt length tells us where to set `max_length`. If a lot of examples
  go above 512 tokens we would need a truncation or windowing strategy, which
  would be a lot of extra work for something that has nothing to do with the
  quantization study.
- the answer length tells us where to set `max_target_length` for generation.

we look at the median and p95 rather than only the max, because a few very long
outliers should not force me to size everything for the worst case.

In [86]:
def describe(name_dataset: str, length_dict: dict):
    for key, item in length_dict.items():
        label, length = key, np.array(item)
    
    if label == 'prompts':
        print(f"Full prompt (question + context) - {name_dataset}:" )
        print(f"share > 512 tokens: {(length > 512).mean():.2%}")
    else:
        print(f"Answer - {name_dataset}:" )
        print(f"share > 32 tokens: {(length > 32).mean():.2%}")

    print(f"median: {np.median(length):.0f} tokens" )
    print(f"mean: {np.mean(length):.0f} tokens" )
    print(f"max: {np.max(length)} tokens" )
    print(f"min: {np.min(length)} tokens" )
    print(f"p95: {np.percentile(length,95):.0f} tokens\n" )

for name_dset, dset in ds.items():
    prompts = [f"question: {q}  context: {c}" for q, c in zip(dset["question"], dset["context"])]
    targets = [target[0] for target in dset["answers"]["text"]]
    prompts_len = [len(prompt) for prompt in tokenizer(prompts).input_ids]    
    targets_len = [len(target) for target in tokenizer(targets).input_ids]
    prompts_len = {'prompts':prompts_len}
    targets_len = {'targets':targets_len}

    describe(name_dset, prompts_len)
    describe(name_dset, targets_len)

Full prompt (question + context) - train:
share > 512 tokens: 0.41%
median: 175 tokens
mean: 189 tokens
max: 843 tokens
min: 40 tokens
p95: 333 tokens

Answer - train:
share > 32 tokens: 2.07%
median: 4 tokens
mean: 7 tokens
max: 386 tokens
min: 2 tokens
p95: 22 tokens

Full prompt (question + context) - val:
share > 512 tokens: 0.43%
median: 174 tokens
mean: 188 tokens
max: 831 tokens
min: 42 tokens
p95: 331 tokens

Answer - val:
share > 32 tokens: 2.10%
median: 4 tokens
mean: 7 tokens
max: 153 tokens
min: 2 tokens
p95: 22 tokens

Full prompt (question + context) - test:
share > 512 tokens: 0.00%
median: 178 tokens
mean: 187 tokens
max: 501 tokens
min: 46 tokens
p95: 317 tokens

Answer - test:
share > 32 tokens: 0.57%
median: 4 tokens
mean: 6 tokens
max: 81 tokens
min: 2 tokens
p95: 15 tokens



Prompts are short: median around 175 tokens and p95 around 330, with only
0.4% above 512. So `max_length = 512` covers basically everything and we don't
need any truncation strategy.

Answers are very short: median 4 tokens, p95 around 22. About 2% go above
32 tokens, which are mostly cases where the annotator highlighted a full
sentence instead of a short span. We can keep `max_target_length = 32` and accept
losing that small tail.

The three splits look consistent (same medians, same means, same p95), so
results should be comparable between them. The test split is slightly cleaner
(shorter answers, nothing above 512) which makes sense since it is the
original validation split of the dataset.

Config kept: `max_length = 512`, `max_target_length = 32`.

### Are the answers really in the context?

The task of this project is extractive: the answer is supposed to be a span taken directly
from the context. But we're using a generative model, which produces free text
instead of pointing at a position in the text.

So we just want to check that every "gold" answer is really a substring of its
context. If some answers were not in the context, they would be impossible to
extract.

In [ ]:
for split, dset in ds.items():
    bad = sum(
        1 for ex in dset
        for t in ex["answers"]["text"]
        if t not in ex["context"]
    )
    print(f"{split}: {bad} answers not found in context")

train: 0 answers not found in context
val: 0 answers not found in context
test: 0 answers not found in context


0 answers not found in the context, on the three splits. 
